# FPN-Mamba Experiments — A100 80GB Colab Runner

**Before starting:** Runtime → Change runtime type → A100 GPU.

**Run order:** Cell 1 → 2 → 3 → 4 → 5 (wandb) → 6 → 7 → 8 → 9 → 10 → 11

## Cell 1 — Verify A100 GPU

In [1]:
import torch

assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> GPU -> A100'
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
bf16_ok  = torch.cuda.is_bf16_supported()

print(f'GPU  : {gpu_name}')
print(f'VRAM : {vram_gb:.1f} GB')
print(f'BF16 : {bf16_ok}  (True = A100 confirmed -- bfloat16 auto-enabled in trainer)')

if not bf16_ok:
    print('WARNING: Not an A100. Reduce batch sizes below if you get OOM.')

GPU  : NVIDIA A100-SXM4-80GB
VRAM : 85.1 GB
BF16 : True  (True = A100 confirmed -- bfloat16 auto-enabled in trainer)


## Cell 2 — Mount Google Drive

Checkpoints and results save here. Code and data come from GitHub, not Drive.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_RESULTS = '/content/drive/MyDrive/fpn_mamba/experiments'
os.makedirs(DRIVE_RESULTS, exist_ok=True)
print('Drive mounted. Results ->', DRIVE_RESULTS)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Results -> /content/drive/MyDrive/fpn_mamba/experiments


## Cell 3 — Clone Repo from GitHub

The dataset (4457 images) is committed in the repo — no separate download needed.

In [ ]:
import os, sys

REPO_URL = 'https://github.com/Tech-sam-90/fpn-inceptentionnet'
REPO_DIR = '/content/fpn-inceptentionnet'
BRANCH   = 'version_2'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

DATA_ROOT = f'{REPO_DIR}/data'
classes   = sorted(d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d)))
n_images  = sum(len(os.listdir(os.path.join(DATA_ROOT, c))) for c in classes)

print(f'Repo    : {REPO_DIR}  (branch: {BRANCH})')
print(f'Data    : {DATA_ROOT}')
print(f'Classes : {classes}')
print(f'Images  : {n_images}')
!git log --oneline -3

## Cell 4 — Install Dependencies

In [ ]:
!pip install -q timm einops scikit-learn scipy thop pyyaml tqdm matplotlib seaborn pandas wandb
print('Done.')

## Cell 5 — Weights & Biases Login

Paste your API key below. **Do not push this cell's key to GitHub.**
This cell runs only in Colab — the key never leaves your session.

In [1]:
import wandb

WANDB_API_KEY = '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'  # <-- replace with your key

wandb.login(key=WANDB_API_KEY, relogin=True)
print('wandb logged in. Project: medulloblastoma-classification')

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sadeniji to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb logged in. Project: medulloblastoma-classification


## Config Helper

Sets A100 80GB batch sizes: InceptentionNet stays at 8 (paper-exact), FPN-Mamba and ablation use 64.

In [2]:
import yaml
from pathlib import Path

def make_config(yaml_path: str, overrides: dict) -> str:
    def _deep_update(base, patch):
        for k, v in patch.items():
            if isinstance(v, dict) and k in base:
                _deep_update(base[k], v)
            else:
                base[k] = v
    with open(yaml_path) as f:
        cfg = yaml.safe_load(f)
    _deep_update(cfg, overrides)
    out = f'/tmp/{Path(yaml_path).stem}_patched.yaml'
    with open(out, 'w') as f:
        yaml.dump(cfg, f)
    return out

# Shared overrides for all experiments
A100_BASE = {
    'data':     {'data_root': DATA_ROOT},
    'training': {'num_workers': 4},  # A100 VMs have 12+ CPU cores
}

print('Config helper ready.')
print('  InceptentionNet batch : 8   (paper-exact, do not change)')
print('  FPN-Mamba batch       : 64  (A100 80GB comfortable at 64)')
print('  Ablation batch        : 64')

NameError: name 'DATA_ROOT' is not defined

## Cell 6 — Train InceptentionNet Baseline

Paper-exact: LR=0.005, **batch=8**, 40 epochs, patience=10, sigma=2.0, 4x augmentation.
Batch size must stay at 8 to faithfully replicate the paper.

In [ ]:
import os

BASELINE_RUN_DIR = f'{DRIVE_RESULTS}/inceptentionnet'
BASELINE_RESULTS = f'{BASELINE_RUN_DIR}/cv_results.json'

if os.path.exists(BASELINE_RESULTS):
    print('Already done. Delete', BASELINE_RESULTS, 'to retrain.')
else:
    cfg = make_config(
        f'{REPO_DIR}/configs/inceptentionnet.yaml',
        overrides={
            **A100_BASE,
            # batch_size stays at 8 -- paper-exact, do not override
            'output': {'run_dir': BASELINE_RUN_DIR},
        }
    )
    !python scripts/train.py --config {cfg}

print('\nBaseline results:', BASELINE_RESULTS)

## Cell 7 — Train FPN-Mamba (Full Model)

A100 80GB: **batch=64**, bfloat16 auto-enabled, num_workers=4.

In [ ]:
import os

FPN_RUN_DIR = f'{DRIVE_RESULTS}/fpn_mamba'
FPN_RESULTS = f'{FPN_RUN_DIR}/cv_results.json'

if os.path.exists(FPN_RESULTS):
    print('Already done. Delete', FPN_RESULTS, 'to retrain.')
else:
    cfg = make_config(
        f'{REPO_DIR}/configs/fpn_mamba.yaml',
        overrides={
            **A100_BASE,
            'training': {'batch_size': 64, 'num_workers': 4},
            'output':   {'run_dir': FPN_RUN_DIR},
        }
    )
    !python scripts/train.py --config {cfg} --baseline_results {BASELINE_RESULTS}

print('\nFPN-Mamba results:', FPN_RESULTS)

## Cell 8 — Ablation Study (5 variants, ~2 hrs total)

Each variant uses **batch=64** on A100 80GB.

In [ ]:
import os

ABL_RUN_DIR = f'{DRIVE_RESULTS}/ablation'
ABL_RESULTS = f'{ABL_RUN_DIR}/ablation_summary.json'

cfg = make_config(
    f'{REPO_DIR}/configs/ablation.yaml',
    overrides={
        **A100_BASE,
        'training': {'batch_size': 64, 'num_workers': 4},
        'output':   {'run_dir': ABL_RUN_DIR},
    }
)
!python scripts/run_ablation.py --config {cfg}

print('\nAblation summary:', ABL_RESULTS)

## Cell 9 — Statistical Comparison (instant)

In [ ]:
import json
from src.evaluation.stats import compare_models, print_comparison_table
from src.evaluation.metrics import summarize_folds

with open(BASELINE_RESULTS) as f: baseline = json.load(f)
with open(FPN_RESULTS)      as f: fpn      = json.load(f)

print('=== InceptentionNet (paper-exact re-run) ===')
for k, v in summarize_folds(baseline['fold_results']).items():
    print(f'  {k:<18}: {v["mean"]:.4f} +/- {v["std"]:.4f}')

print('\n=== FPN-Mamba ===')
for k, v in summarize_folds(fpn['fold_results']).items():
    print(f'  {k:<18}: {v["mean"]:.4f} +/- {v["std"]:.4f}')

print('\n=== Statistical Comparison (bootstrap 95% CI + Wilcoxon p) ===')
table = compare_models(
    baseline['fold_results'], fpn['fold_results'],
    name_a='InceptentionNet', name_b='FPN-Mamba'
)
print_comparison_table(table, name_a='InceptentionNet', name_b='FPN-Mamba')

## Cell 10 — Ablation Table (instant)

In [ ]:
import json, pandas as pd

with open(ABL_RESULTS) as f:
    abl = json.load(f)

DISPLAY = {
    'efficientnet_only': 'EfficientNet-B2 only',
    'fpn_standard':      '+ FPN (standard 3x3)',
    'fpn_locality':      '+ LocalityMixing',
    'fpn_cross_mamba':   '+ Cross-scale Mamba',
    'fpn_mamba_full':    '+ GeM + SE  (full model)',
}
METRICS = ['accuracy', 'precision', 'recall', 'sensitivity', 'specificity', 'f1', 'auc']

rows = []
for variant, name in DISPLAY.items():
    if variant not in abl: continue
    row = {'Variant': name}
    for m in METRICS:
        mu  = abl[variant].get(m, {}).get('mean', float('nan'))
        std = abl[variant].get(m, {}).get('std',  float('nan'))
        row[m] = f'{mu:.4f} +/- {std:.4f}'
    rows.append(row)

df = pd.DataFrame(rows).set_index('Variant')
print(df.to_string())

## Cell 11 — Verify Drive (all checkpoints saved)

In [ ]:
import os

def check(path, label):
    ok   = os.path.exists(path)
    size = f'{os.path.getsize(path)/1024:.0f} KB' if ok else ''
    print(f"  {'[OK]' if ok else '[MISSING]':<10} {label:<40} {size}")

print('=== Drive Contents ===')
check(BASELINE_RESULTS, 'InceptentionNet cv_results.json')
check(FPN_RESULTS,      'FPN-Mamba cv_results.json')
check(ABL_RESULTS,      'Ablation summary.json')
for run_name, run_dir in [('Baseline', BASELINE_RUN_DIR), ('FPN-Mamba', FPN_RUN_DIR)]:
    for fold in range(1, 6):
        check(f'{run_dir}/fold_{fold}.pt', f'{run_name} fold_{fold}.pt')
    check(f'{run_dir}/best_model.pt', f'{run_name} best_model.pt')